# Fine-tune RF-DETR on the NEON tree-crown benchmark

Same T4 workflow and the same `urban_canopy` scoring as the YOLO26 notebook, using the **COCO** dataset layout. RF-DETR is the current real-time DETR-class reference; results are written to the shared `results/metrics.json` so the README table compares all models on identical splits.

In [ ]:
# --- Environment (Colab/Kaggle T4) ---
# Clone the repo so the tested urban_canopy package drives data + metrics.
import os
import subprocess
import sys

if not os.path.exists("urban-canopy-detection"):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/tarpous/urban-canopy-detection.git"],
        check=True,
    )
%cd urban-canopy-detection
sys.path.insert(0, "src")

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "-q",
        "install",
        "rfdetr",
        "sahi",
        "pillow",
        "pyyaml",
        "rasterio",
        "geopandas",
    ],
    check=True,
)

In [ ]:
# --- Data: download NEON evaluation split, tile, build a site-disjoint dataset ---
# ~3.9 GB download; the evaluation split doubles as train/val here via geographic
# blocking (the benchmark's own train set is optional and much larger).
subprocess.run(
    [sys.executable, "scripts/download_neon.py", "--evaluation", "--annotations"], check=True
)

from pathlib import Path

from urban_canopy.dataset import build_coco_dataset, discover_pairs

pairs = discover_pairs(Path("data/raw/evaluation/RGB"), Path("data/raw/annotations"))
print(f"{len(pairs)} annotated tiles across {len({p[0].name.split('_')[1] for p in pairs})} sites")

In [ ]:
MODEL_NAME, SLUG = "RF-DETR (fine-tuned)", "rfdetr"
manifest = build_coco_dataset(
    pairs,
    Path("data/coco"),
    val_fraction=0.2,
    seed=0,
    tile_size=640,
    overlap=128,
    min_visibility=0.3,
)
print(manifest)
assert not (set(manifest.train_sites) & set(manifest.val_sites)), "site leakage!"

In [ ]:
# --- Train ---
from rfdetr import RFDETRBase

model = RFDETRBase()
model.train(dataset_dir="data/coco", epochs=60, batch_size=8, lr=1e-4, output_dir=f"runs/{SLUG}")

In [ ]:
# --- SAHI sliced inference over the validation images ---
from PIL import Image

from urban_canopy.sliced import sliced_predict
from urban_canopy.tiling import TileWindow


def make_detector(image):
    def detect(window: TileWindow):
        crop = image.crop((window.x0, window.y0, window.x1, window.y1))
        preds = model.predict(crop, threshold=0.05)
        dets = []
        for box, score in zip(preds.xyxy, preds.confidence):
            dets.append(Detection(Box(*[float(v) for v in box]), float(score)))
        return dets

    return detect


predictions_per_image = {}
for image_path in sorted(Path("data/coco/valid").glob("*.png")):
    with Image.open(image_path) as im:
        w, h = im.size
        predictions_per_image[image_path.name] = sliced_predict(
            w, h, make_detector(im), tile_size=640, overlap=128
        )
print(f"{sum(len(v) for v in predictions_per_image.values())} detections")

In [ ]:
# --- Score with the repo's evaluator and write results/metrics.json ---
import json

from urban_canopy.evaluate import Detection
from urban_canopy.labels import Box
from urban_canopy.predictions import (
    detections_to_json,
    load_ground_truth,
    result_to_model_row,
    score_predictions,
)

# predictions_per_image: dict[str, list[Detection]] built in the inference cell above
ground_truth = load_ground_truth(Path("data/raw/annotations"))
result = score_predictions(predictions_per_image, ground_truth, score_threshold=0.3)
print(result)

metrics = json.loads(Path("results/metrics.json").read_text())
row = result_to_model_row(MODEL_NAME, result, inference="SAHI sliced (640/128)")
metrics["models"] = [
    m for m in metrics["models"] if not m["name"].startswith(MODEL_NAME.split()[0])
]
metrics["models"].append(row)
metrics["dataset"]["status"] = "measured"
Path("results/metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
Path(f"results/{SLUG}_predictions.json").write_text(detections_to_json(predictions_per_image))
print("wrote results/metrics.json and predictions — commit these back to the repo.")